In [32]:
# !pip install .

In [34]:
import numpy as np
import pandas as pd
from statsmodels.robust import mad
import os
import time

from sklearn.linear_model import LassoLarsIC, Lasso, LassoCV

from epftoolbox.data import scaling
from epftoolbox.data import read_data
from epftoolbox.evaluation import MAE, sMAPE

from sklearn.utils._testing import ignore_warnings
from sklearn.exceptions import ConvergenceWarning

In [35]:
class LEAR(object):
    """Class to build a LEAR model, recalibrate it, and use it to predict DA electricity prices.
    
    An example on how to use this class is provided :ref:`here<learex2>`.
    
    Parameters
    ----------
    calibration_window : int, optional
        Calibration window (in days) for the LEAR model.
        
    """
    
    def __init__(self, calibration_window=364 * 3):

        # Calibration window in hours
        self.calibration_window = calibration_window
        self._const_idx   = None   
        self._center_idx  = None   
        self._scale_idx   = None 
        #self._dup_idx = None

    # Ignore convergence warnings from scikit-learn LASSO module
    @ignore_warnings(category=ConvergenceWarning)
    def recalibrate(self, Xtrain, Ytrain):
        """Function to recalibrate the LEAR model. 
        
        It uses a training (Xtrain, Ytrain) pair for recalibration
        
        Parameters
        ----------
        Xtrain : numpy.array
            Input in training dataset. It should be of size *[n,m]* where *n* is the number of days
            in the training dataset and *m* the number of input features
        
        Ytrain : numpy.array
            Output in training dataset. It should be of size *[n,24]* where *n* is the number of days 
            in the training dataset and 24 are the 24 prices of each day
                
        Returns
        -------
        numpy.array
            The prediction of day-ahead prices after recalibrating the model        
        
        """

        # # Applying Invariant, aka asinh-median transformation to the prices
        [Ytrain], self.scalerY = scaling([Ytrain], 'Invariant')
        #self.scalerY = True ## for congestion
        
        # # Rescaling all inputs except dummies (7 last features)
        # [Xtrain_no_dummies], self.scalerX = scaling([Xtrain[:, :-7]], 'Invariant')
        # Xtrain[:, :-7] = Xtrain_no_dummies
        
        #self.scalerX = None

        ## delete constant column if have
        
        non_dummy = np.arange(Xtrain.shape[1] - 7)             # last 7 weekday dummies
        const_mask = np.ptp(Xtrain[:, non_dummy], axis=0) == 0
        self._const_idx = non_dummy[const_mask]                
        
        if self._const_idx.size:
            Xtrain = np.delete(Xtrain, self._const_idx, axis=1)
        
        # 
        non_dummy_new = np.arange(Xtrain.shape[1] - 7)
        centered = Xtrain - Xtrain.mean(axis=0, keepdims=True)
        center_mask = np.ptp(centered[:, non_dummy_new], axis=0) == 0
        self._center_idx = non_dummy_new[center_mask]          
        
        if self._center_idx.size:
            Xtrain = np.delete(Xtrain, self._center_idx, axis=1)
            
        # non_dummy_dup = np.arange(Xtrain.shape[1] - 7)
        # df_dup = pd.DataFrame(Xtrain[:, non_dummy_dup])
        # dup_mask = df_dup.T.duplicated().to_numpy()
        # self._dup_idx = non_dummy_dup[dup_mask]
        # if self._dup_idx.size:
        #     Xtrain = np.delete(Xtrain, self._dup_idx, axis=1)
        # else:
        #     self._dup_idx = None
        #  only scale for mad not equal to 0
        non_dummy_final = np.arange(Xtrain.shape[1] - 7)
        safe_cols = [i for i in non_dummy_final if mad(Xtrain[:, i]) != 0]
        self._scale_idx = np.array(safe_cols, dtype=int)

        print("Deleted constant columns (index):", self._const_idx)
        print("Deleted centered-constant columns (index):", self._center_idx)
        
        if self._scale_idx.size:
            [X_scaled], self.scalerX = scaling([Xtrain[:, self._scale_idx]], 'Invariant')
            Xtrain[:, self._scale_idx] = np.nan_to_num(X_scaled, nan=0.0)
        else:
            self.scalerX = None



        
        # print("\n Checking MAD == 0 columns in Xtrain:")
        # mad_vals = self.scalerX.scaler.mad
        # zero_mad_cols = np.where(mad_vals == 0)[0]
        # print(" MAD = 0 at columns:", zero_mad_cols)

        # self.models = {}
        # for h in range(24):

        #     # Estimating lambda hyperparameter using LARS
        #     param_model = LassoLarsIC(criterion='aic', max_iter=2500)
        #     param = param_model.fit(Xtrain, Ytrain[:, h]).alpha_

        #     # Re-calibrating LEAR using standard LASSO estimation technique
        #     model = Lasso(max_iter=2500, alpha=param)
        #     model.fit(Xtrain, Ytrain[:, h])

        #     self.models[h] = model

        alpha_24 = np.zeros(24)
        coef_24 = np.zeros((24, 370))
        
        self.models = {}
        for h in range(24):
            y_h = Ytrain[:, h]
            
            try:
                # ===== 原做法 =====
                param_model = LassoLarsIC(criterion='aic', max_iter=2500)
                alpha = param_model.fit(Xtrain, y_h).alpha_
                model = Lasso(max_iter=2500, alpha=alpha).fit(Xtrain, y_h)
                alpha_24[h] = alpha
                coef_24[h, :] = model.coef_
                
            except ValueError as e:

                if "operands could not be broadcast" in str(e):
                    print(f"[WARN] LARS failed on hour {h}. Falling back to LassoCV.")
                    model = LassoCV(cv=5, max_iter=2500, n_jobs=-1).fit(Xtrain, y_h)
                else:
                    raise

            self.models[h] = model

        return alpha_24, coef_24 
        
    def predict(self, X):
        """Function that makes a prediction using some given inputs.
        
        Parameters
        ----------
        X : numpy.array
            Input of the model.
        
        Returns
        -------
        numpy.array
            An array containing the predictions.
        """

        # Predefining predicted prices
        Yp = np.zeros(24)

        # # Rescaling all inputs except dummies (7 last features)
        # X_no_dummies = self.scalerX.transform(X[:, :-7])
        # X[:, :-7] = X_no_dummies

        # ---- 与训练阶段相同的列做同样缩放 ----
        # --- 先按训练阶段的两级索引删列 ---
        X_tmp = np.delete(X,   self._const_idx,  axis=1) if self._const_idx  is not None else X.copy()
        X_tmp = np.delete(X_tmp, self._center_idx, axis=1) if self._center_idx is not None else X_tmp
        #X_tmp = np.delete(X_tmp, self._dup_idx,    axis=1) if self._dup_idx    is not None else X_tmp
        # --- 再做与训练相同的 Invariant 缩放 ---
        if self.scalerX is not None and self._scale_idx is not None and self._scale_idx.size:
            X_tmp[:, self._scale_idx] = self.scalerX.transform(X_tmp[:, self._scale_idx])




        # Predicting the current date using a recalibrated LEAR
        
        for h in range(24):

            # Predicting test dataset and saving
            Yp[h] = self.models[h].predict(X_tmp)[0]
            #print(f"Hour {h}, prediction shape: {self.models[h].predict(X).shape}, value: {self.models[h].predict(X)}")
        
        Yp = self.scalerY.inverse_transform(Yp.reshape(1, -1))
        Yp = Yp.reshape(1,-1)

        return Yp

    def recalibrate_predict(self, Xtrain, Ytrain, Xtest):
        """Function that first recalibrates the LEAR model and then makes a prediction.

        The function receives the training dataset, and trains the LEAR model. Then, using
        the inputs of the test dataset, it makes a new prediction.
        
        Parameters
        ----------
        Xtrain : numpy.array
            Input of the training dataset.
        Xtest : numpy.array
            Input of the test dataset.
        Ytrain : numpy.array
            Output of the training dataset.
        
        Returns
        -------
        numpy.array
            An array containing the predictions in the test dataset.
        """

        alpha_24, coef_24 = self.recalibrate(Xtrain=Xtrain, Ytrain=Ytrain)    

        Yp = self.predict(X=Xtest)

        return alpha_24, coef_24, Yp

    def _build_and_split_XYs(self, df_train, df_test=None, date_test=None, naive_exog=False):
        
        """Internal function that generates the X,Y arrays for training and testing based on pandas dataframes
        
        Parameters
        ----------
        df_train : pandas.DataFrame
            Pandas dataframe containing the training data
        
        df_test : pandas.DataFrame
            Pandas dataframe containing the test data
        
        date_test : datetime, optional
            If given, then the test dataset is only built for that date
        
        Returns
        -------
        list
            [Xtrain, Ytrain, Xtest] as the list containing the (X,Y) input/output pairs for training, 
            and the input for testing
        """
        
        # Checking that the first index in the dataframes corresponds with the hour 00:00 
        if df_train.index[0].hour != 0 or df_test.index[0].hour != 0:
            print('Problem with the index')

        # 
        # Defining the number of Exogenous inputs
        n_exogenous_inputs = len(df_train.columns) - 1

        # 96 prices + n_exogenous * (24 * 3 exogeneous) + 7 weekday dummies
        # Price lags: D-1, D-2, D-3, D-7
        # Exogeneous inputs lags: D, D-1, D-7
        n_features = 24*4 + 7 + n_exogenous_inputs * 24*3
        # 加 4 维 Fourier（日内 2 + 周内 2）和 1 维趋势
        # n_fourier = 4
        # n_trend   = 1
        # n_features = 24*4 + 7 + n_exogenous_inputs * 24*3 + n_fourier + n_trend
    
        # Extracting the predicted dates for testing and training. We leave the first week of data
        # out of the prediction as we the maximum lag can be one week
        
        # We define the potential time indexes that have to be forecasted in training
        # and testing
        indexTrain = df_train.loc[df_train.index[0] + pd.Timedelta(weeks=1):].index ## contain 1 week lag / week =1 

        # For testing, the test dataset is different whether depending on whether a specific test
        # dataset is provided
        if date_test is None:
            indexTest = df_test.loc[df_test.index[0] + pd.Timedelta(weeks=1):].index ## no specified, just based on the date in the df_test
        ## and forecast to the end. So df_test should include up to a week ago data.
        else:
            indexTest = df_test.loc[date_test:date_test + pd.Timedelta(hours=23)].index ## specified, test for that day

        # We extract the prediction dates/days.
        predDatesTrain = indexTrain.round('1h')[::24]                
        predDatesTest = indexTest.round('1h')[::24]
        print(predDatesTest)
        # We create two dataframe to build XY.
        # These dataframes have as indices the first hour of the day (00:00)
        # and the columns represent the 23 possible horizons/dates along a day
        indexTrain = pd.DataFrame(index=predDatesTrain, columns=['h' + str(hour) for hour in range(24)])
        indexTest = pd.DataFrame(index=predDatesTest, columns=['h' + str(hour) for hour in range(24)])
        for hour in range(24):
            indexTrain.loc[:, 'h' + str(hour)] = indexTrain.index + pd.Timedelta(hours=hour)
            indexTest.loc[:, 'h' + str(hour)] = indexTest.index + pd.Timedelta(hours=hour)

        #print(indexTest)
        # Preallocating in memory the X and Y arrays          
        Xtrain = np.zeros([indexTrain.shape[0], n_features])
        Xtest = np.zeros([indexTest.shape[0], n_features])
        Ytrain = np.zeros([indexTrain.shape[0], 24])

        # --- 插入开始：平滑周期 & 趋势特征 ---
        # 训练集上按日（24h）和周（7d）做 Fourier，再加一个归一化天数趋势
        # hours_train      = indexTrain.index.hour.values
        # dows_train       = indexTrain.index.dayofweek.values
        # days_since_train = (indexTrain.index - indexTrain.index[0]).days.values
        # fourier_train = np.column_stack([
        #     np.sin(2*np.pi * hours_train / 24),
        #     np.cos(2*np.pi * hours_train / 24),
        #     np.sin(2*np.pi * dows_train  / 7),
        #     np.cos(2*np.pi * dows_train  / 7),
        # ])
        # trend_train = (days_since_train / days_since_train.max()).reshape(-1, 1)

        # # 测试集同理
        # hours_test       = indexTest.index.hour.values
        # dows_test        = indexTest.index.dayofweek.values
        # days_since_test  = (indexTest.index - indexTrain.index[0]).days.values
        # fourier_test = np.column_stack([
        #     np.sin(2*np.pi * hours_test / 24),
        #     np.cos(2*np.pi * hours_test / 24),
        #     np.sin(2*np.pi * dows_test  / 7),
        #     np.cos(2*np.pi * dows_test  / 7),
        # ])
        # trend_test = (days_since_test / days_since_train.max()).reshape(-1, 1)

        # # 把它们塞进 Xtrain/Xtest 的最后 5 列（确保 n_features 里已加 5）
        # Xtrain[:, -5:-1] = fourier_train
        # Xtrain[:,    -1] = trend_train.squeeze()

        # Xtest[:,  -5:-1] = fourier_test
        # Xtest[:,   -1] = trend_test.squeeze()


        # Index that 
        feature_index = 0
        
        #
        # Adding the historial prices during days D-1, D-2, D-3, and D-7
        #

        # For each hour of a day
        for hour in range(24):
            # For each possible past day where prices can be included
            for past_day in [1, 2, 3,7]:

                # We define the corresponding past time indexs using the auxiliary dataframses 
                pastIndexTrain = pd.to_datetime(indexTrain.loc[:, 'h' + str(hour)].values) - \
                    pd.Timedelta(hours=24 * past_day)
                pastIndexTest = pd.to_datetime(indexTest.loc[:, 'h' + str(hour)].values) - \
                    pd.Timedelta(hours=24 * past_day)

                # We include the historical prices at day D-past_day and hour "h" 
                Xtrain[:, feature_index] = df_train.loc[pastIndexTrain, 'Price']
                Xtest[:, feature_index] = df_test.loc[pastIndexTest, 'Price']
                feature_index += 1

        #
        # Adding the exogenous inputs during days D, D-1,  D-7
        #
        # For each hour of a day
        for hour in range(24):
            # For each possible past day where exogenous inputs can be included
            for past_day in [1, 7]:
                # For each of the exogenous input
                for exog in range(1, n_exogenous_inputs + 1):

                    # Definying the corresponding past time indexs using the auxiliary dataframses 
                    pastIndexTrain = pd.to_datetime(indexTrain.loc[:, 'h' + str(hour)].values) - \
                        pd.Timedelta(hours=24 * past_day)
                    pastIndexTest = pd.to_datetime(indexTest.loc[:, 'h' + str(hour)].values) - \
                        pd.Timedelta(hours=24 * past_day)

                    # Including the exogenous input at day D-past_day and hour "h" 
                    Xtrain[:, feature_index] = df_train.loc[pastIndexTrain, 'Exogenous ' + str(exog)]                    
                    Xtest[:, feature_index] = df_test.loc[pastIndexTest, 'Exogenous ' + str(exog)]
                    feature_index += 1

            # For each of the exogenous inputs we include feature if feature selection indicates it
        # for hour in range(24):
        #     for exog in range(1, n_exogenous_inputs + 1):
        #         futureIndexTrain = pd.to_datetime(indexTrain.loc[:, 'h' + str(hour)].values)
        #         # For test set, if naive_exog is True, use previous day's 23:00 value as constant
        #         if naive_exog:
        #             # Determine the constant time: for the forecast day, use the previous day at 23:00
        #             # Here date_test is the forecast day start (if provided) or derived from indexTest
        #             if date_test is not None:
        #                 const_time = (pd.to_datetime(date_test) - pd.Timedelta(days=1)).replace(hour=23, minute=0, second=0, microsecond=0)
        #             else:
        #                 # If no specific date_test provided, take the first day in indexTest and subtract one day
        #                 const_time = (pd.to_datetime(indexTest.index[0]) - pd.Timedelta(days=1)).replace(hour=23, minute=0, second=0, microsecond=0)
        #             # For Xtest, use constant value from const_time
        #             if const_time in df_test.index:
        #                 constant_value = df_test.loc[const_time, 'Exogenous ' + str(exog)]
        #             else:
        #                 constant_value = df_train.loc[const_time, 'Exogenous ' + str(exog)]
        #             Xtest[:, feature_index] = constant_value
        #         else:
        #             futureIndexTest = pd.to_datetime(indexTest.loc[:, 'h' + str(hour)].values)
        #             Xtest[:, feature_index] = df_test.loc[futureIndexTest, 'Exogenous ' + str(exog)]
        #         # For training, always use the actual value at day D
        #         Xtrain[:, feature_index] = df_train.loc[futureIndexTrain, 'Exogenous ' + str(exog)]
        #         feature_index += 1
        for hour in range(24):
            for exog in range(1, n_exogenous_inputs + 1):
                futureIndexTrain = pd.to_datetime(indexTrain.loc[:, 'h' + str(hour)].values)
                Xtrain[:, feature_index] = df_train.loc[futureIndexTrain, 'Exogenous ' + str(exog)]

    
                futureIndexTest = pd.to_datetime(indexTest.loc[:, 'h' + str(hour)].values)

                naive_feature_names = ['Exogenous 4']  # ['Exogenous 1', 'Exogenous 3']

                exog_name = 'Exogenous ' + str(exog)
                if naive_exog and exog_name in naive_feature_names:
                    Xtest[:, feature_index] = df_test.loc[
                        futureIndexTest - pd.Timedelta(hours=24),
                        exog_name
                    ]
                else:
                    Xtest[:, feature_index] = df_test.loc[
                        futureIndexTest,
                        exog_name
                    ]

                feature_index += 1

        #
        # Adding the dummy variables that depend on the day of the week. Monday is 0 and Sunday is 6
        #
        # For each day of the week
        for dayofweek in range(7):
            Xtrain[indexTrain.index.dayofweek == dayofweek, feature_index] = 1
            Xtest[indexTest.index.dayofweek == dayofweek, feature_index] = 1
            feature_index += 1

        # Extracting the predicted values Y
        for hour in range(24):
            # Defining time index at hour h
            futureIndexTrain = pd.to_datetime(indexTrain.loc[:, 'h' + str(hour)].values)
            futureIndexTest = pd.to_datetime(indexTest.loc[:, 'h' + str(hour)].values)

            # Extracting Y value based on time indexs
            Ytrain[:, hour] = df_train.loc[futureIndexTrain, 'Price']     

        # print("Naive forecast for exogenous variables (sample) in Xtest:")
        # print(Xtest[0, 576:675])

        return Xtrain, Ytrain, Xtest


    def recalibrate_and_forecast_next_day(self, df, calibration_window, next_day_date,naive_exog=False):
        """Easy-to-use interface for daily recalibration and forecasting of the LEAR model.
        
        The function receives a pandas dataframe and a date. Usually, the data should
        correspond with the date of the next-day when using for daily recalibration.
        
        Parameters
        ----------
        df : pandas.DataFrame
            Dataframe of historical data containing prices and *N* exogenous inputs. 
            The index of the dataframe should be dates with hourly frequency. The columns 
            should have the following names ``['Price', 'Exogenous 1', 'Exogenous 2', ...., 'Exogenous N']``.
        
        calibration_window : int
            Calibration window (in days) for the LEAR model.
        
        next_day_date : datetime
            Date of the day-ahead.
        
        Returns
        -------
        numpy.array
            The prediction of day-ahead prices.
        """

        # We define the new training dataset and test datasets 
        df_train = df.loc[:next_day_date - pd.Timedelta(hours=1)]
        # Limiting the training dataset to the calibration window
        df_train = df_train.iloc[-calibration_window * 24:]
    
        # We define the test dataset as the next day (they day of interest) plus the last two weeks
        # in order to be able to build the necessary input features. 
        df_test = df.loc[next_day_date - pd.Timedelta(weeks=2):, :] ## 2


        # Generating X,Y pairs for predicting prices
        Xtrain, Ytrain, Xtest, = self._build_and_split_XYs(
            df_train=df_train, df_test=df_test, date_test=next_day_date,naive_exog = naive_exog)

        # Recalibrating the LEAR model and extracting the prediction
        alpha_24, coef_24, Yp = self.recalibrate_predict(Xtrain=Xtrain, Ytrain=Ytrain, Xtest=Xtest)

        return alpha_24, coef_24, Yp

In [36]:
def evaluate_lear_in_test_dataset(path_datasets_folder=os.path.join('.', 'datasets'), 
                                  path_recalibration_folder=os.path.join('.', 'experimental_files'),
                                  dataset='PJM', years_test=2, calibration_window=364 * 3, 
                                  begin_test_date=None, end_test_date=None,naive_exog=False):
    """Function for easy evaluation of the LEAR model in a test dataset using daily recalibration. 
    
    The test dataset is defined by a market name and the test dates dates. The function
    generates the test and training datasets, and evaluates a LEAR model considering daily recalibration. 
    
    An example on how to use this function is provided :ref:`here<learex1>`.   

    Parameters
    ----------
    path_datasets_folder : str, optional
        path where the datasets are stored or, if they do not exist yet,
        the path where the datasets are to be stored.
    
    path_recalibration_folder : str, optional
        path to save the files of the experiment dataset.
    
    dataset : str, optional
        Name of the dataset/market under study. If it is one one of the standard markets, 
        i.e. ``"PJM"``, ``"NP"``, ``"BE"``, ``"FR"``, or ``"DE"``, the dataset is automatically downloaded. If the name
        is different, a dataset with a csv format should be place in the ``path_datasets_folder``.

    years_test : int, optional
        Number of years (a year is 364 days) in the test dataset. It is only used if 
        the arguments ``begin_test_date`` and ``end_test_date`` are not provided.
    
    calibration_window : int, optional
        Number of days used in the training dataset for recalibration.
    
    begin_test_date : datetime/str, optional
        Optional parameter to select the test dataset. Used in combination with the argument
        ``end_test_date``. If either of them is not provided, the test dataset is built using the 
        ``years_test`` argument. ``begin_test_date`` should either be a string with the following 
        format ``"%d/%m/%Y %H:%M"``, or a datetime object.
    
    end_test_date : datetime/str, optional
        Optional parameter to select the test dataset. Used in combination with the argument
        ``begin_test_date``. If either of them is not provided, the test dataset is built using the 
        ``years_test`` argument. ``end_test_date`` should either be a string with the following 
        format ``"%d/%m/%Y %H:%M"``, or a datetime object.       
    
    Returns
    -------
    pandas.DataFrame
        A dataframe with all the predictions in the test dataset. The dataframe is also written to path_recalibration_folder.
    """

    # Checking if provided directory for recalibration exists and if not create it
    if not os.path.exists(path_recalibration_folder):
        os.makedirs(path_recalibration_folder)

    # Defining train and testing data
    df_train, df_test = read_data(dataset=dataset, years_test=years_test, path=path_datasets_folder,
                                  begin_test_date=begin_test_date, end_test_date=end_test_date)
    #print(df_test)
    # Defining unique name to save the forecast
    forecast_file_name = 'LEAR_forecast' + '_dat' + str(dataset) + '_YT' + str(years_test) + \
                         '_CW' + str(calibration_window) + '.csv'

    forecast_file_path = os.path.join(path_recalibration_folder, forecast_file_name)


    # Defining empty forecast array and the real values to be predicted in a more friendly format
    forecast = pd.DataFrame(index=df_test.index[::24], columns=['h' + str(k) for k in range(24)])
    real_values = df_test.loc[:, ['Price']].values.reshape(-1, 24)
    print("真实值:", real_values)
    real_values = pd.DataFrame(real_values, index=forecast.index, columns=forecast.columns)
    

    forecast_dates = forecast.index

    model = LEAR(calibration_window=calibration_window)

    alpha_list = []
    coef_list = []
    
    # For loop over the recalibration dates
    for date in forecast_dates:

        start = time.time()
        
        # For simulation purposes, we assume that the available data is
        # the data up to current date where the prices of current date are not known
        data_available = pd.concat([df_train, df_test.loc[:date + pd.Timedelta(hours=23), :]], axis=0)

        # We set the real prices for current date to NaN in the dataframe of available data
        data_available.loc[date:date + pd.Timedelta(hours=23), 'Price'] = np.nan
        # print("当前预测日期：", date)
        # print("用于预测的数据日期范围：", data_available.index.min(), "至", data_available.index.max())
    

        # Recalibrating the model with the most up-to-date available data and making a prediction
        # for the next day
        
        alpha_24, coef_24, Yp = model.recalibrate_and_forecast_next_day(df=data_available, next_day_date=date, 
                                                     calibration_window=calibration_window,naive_exog = True)
        alpha_list.append(alpha_24)
        coef_list.append(coef_24)
        
        true_vals = real_values.loc[date].values
        #print("真实值:", true_vals)
        # Saving the current prediction
        forecast.loc[date, :] = Yp

        # Computing metrics up-to-current-date
        mae = np.mean(MAE(forecast.loc[:date].values.squeeze(), real_values.loc[:date].values)) 
        smape = np.mean(sMAPE(forecast.loc[:date].values.squeeze(), real_values.loc[:date].values)) * 100
        mae_current = MAE(forecast.loc[date].values.squeeze(), real_values.loc[date].values) 
        # Pringint information
        print('{} - sMAPE: {:.2f}%  |  MAE: {:.3f} | MAE_c : {:.3f}'.format(str(date)[:10], smape, mae, mae_current))

        end = time.time()
        print(f"运行时间：{end - start:.4f} 秒")
        
        # Saving forecast
        forecast.to_csv(forecast_file_path)

    return alpha_list, coef_list, forecast

In [50]:
# 假设 evaluate_lear_in_test_dataset 函数已经导入或定义在当前环境中

datasets_folder = os.path.join('.', 'datasets')
#recalib_folder = os.path.join('.', 'experimental_files') ## for first case, assume the exogenous are known for forecast horizon.
recalib_folder = os.path.join('.', 'experimental_files_naive_multivariate')

begin_test_date = '31/12/2024 00:00'
end_test_date   = '31/12/2024 23:00'

In [51]:
total_start = time.time()
alpha_list, coef_list, forecast_df = evaluate_lear_in_test_dataset(
    path_datasets_folder=datasets_folder,
    path_recalibration_folder=recalib_folder,
    dataset='lmp_NP15_data_new',              # 这里 "mydata" 为你的数据集名称，可根据需要修改
    #years_test=2,                  # 如果你已经指定了测试集时间，这个参数会被忽略
    calibration_window=1092,    # 使用三年的数据作为校准窗口
    begin_test_date=begin_test_date,
    end_test_date=end_test_date,
    naive_exog = True #Fuel Price need too use naive forecast
)

# 输出预测结果
#print("预测结果:")
#print(forecast_df)
total_end = time.time()
print(f"运行时间：{total_end - total_start:.4f} 秒")

Test datasets: 2024-12-31 00:00:00 - 2024-12-31 23:00:00
真实值: [[42.44088 46.01963 42.83388 42.51278 45.3963  50.52266 53.00938 52.57481
  42.67195 36.7989  35.90472 34.94792 32.39658 31.62677 34.82989 39.5525
  53.67586 54.70905 52.57391 52.9951  52.47223 52.25263 51.77844 51.34691]]
DatetimeIndex(['2024-12-31'], dtype='datetime64[ns]', freq=None)
Deleted constant columns (index): [ 97 101 105 109 113 117 121 125 129 133 273 277 281 285 289 293 297 301
 305 377 381]
Deleted centered-constant columns (index): []
2024-12-31 - sMAPE: 17.73%  |  MAE: 7.393 | MAE_c : 7.393
运行时间：3.5860 秒
运行时间：4.1107 秒


In [39]:
start_datetime = pd.to_datetime(begin_test_date, format='%d/%m/%Y %H:%M')

In [40]:
alpha_array = np.array(alpha_list)
num_days, num_hours = alpha_array.shape
alpha_reshaped = alpha_array.reshape(num_days * num_hours)
# 生成时间序列索引（每小时一个点）
time_index = pd.date_range(start=start_datetime, periods=num_days * num_hours, freq='H')
# 生成特征列名
feature_names = ['alpha']
# 创建 DataFrame
alpha_df = pd.DataFrame(alpha_reshaped, columns=feature_names)
alpha_df['datetime'] = time_index

# 把 datetime 放最前面
cols = ['datetime'] + feature_names
alpha_df = alpha_df[cols]

/var/folders/jp/nl85t4mj1118k5q8j1b1sdkm0000gn/T/ipykernel_17630/1820626865.py:5: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  time_index = pd.date_range(start=start_datetime, periods=num_days * num_hours, freq='H')


In [41]:
coef_array = np.array(coef_list)
num_days, num_hours, num_features = coef_array.shape
coef_reshaped = coef_array.reshape(num_days * num_hours, num_features)
# 生成时间序列索引（每小时一个点）
time_index = pd.date_range(start=start_datetime, periods=num_days * num_hours, freq='H')
# 生成特征列名
feature_names = [f'feature_{i}' for i in range(num_features)]
# 创建 DataFrame
coef_df = pd.DataFrame(coef_reshaped, columns=feature_names)
coef_df['datetime'] = time_index

# 把 datetime 放最前面
cols = ['datetime'] + feature_names
coef_df = coef_df[cols]

/var/folders/jp/nl85t4mj1118k5q8j1b1sdkm0000gn/T/ipykernel_17630/2708265466.py:5: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  time_index = pd.date_range(start=start_datetime, periods=num_days * num_hours, freq='H')


In [42]:
forecast_df.to_csv('lear_lmp_BANC.csv')
alpha_df.to_csv('lear_lmp_alpha_BANC.csv')
coef_df.to_csv('lear_lmp_coef_BANC.csv')

In [43]:
pred = pd.read_csv("lear_lmp_BANC.csv")
pred = pred.rename(columns={"Unnamed: 0": "date"})
pred["date"] = pd.to_datetime(pred["date"])

df_long = pred.melt(
    id_vars="date",
    var_name="hour",
    value_name="value"
)

# extract hour number from h0, h1, ...
df_long["hour"] = df_long["hour"].str.extract(r"(\d+)").astype(int)

# build timestamp (h0 = 00:00)
df_long["timestamp"] = (
    df_long["date"]
    + pd.to_timedelta(df_long["hour"], unit="h")
)

# final cleanup
df_long = df_long.sort_values("timestamp").reset_index(drop=True)
df_long.head()
truth = pd.read_csv("datasets/lmp_NP15_data.csv")
truth = truth.rename(columns={"Unnamed: 0": "timestamp"})
start = "2025-01-01 00:00"
end   = "2026-01-01 00:00"

truth["timestampt"] = pd.to_datetime(truth["timestamp"])
test_df = truth[(truth["timestamp"] >= start) & (truth["timestamp"] < end)]
test_df.head()
df_long["timestamp"] = pd.to_datetime(df_long["timestamp"])
test_df["timestamp"] = pd.to_datetime(test_df["timestamp"])


result_df = df_long.merge(
    test_df[["timestamp", "Price"]],
    on=[ "timestamp"],
    how="inner"
)

result_df.head()
from sklearn.metrics import mean_absolute_percentage_error, root_mean_squared_error, mean_pinball_loss, mean_absolute_error

pred = result_df['value'].copy().to_numpy()
target = result_df['Price'].copy().to_numpy()

mape_val = mean_absolute_percentage_error(pred, target)
rmse_val = root_mean_squared_error(pred, target)
mae_val = mean_absolute_error(pred, target)
# pl_list=[]
# for q in quantiles:
#   pl = mean_pinball_loss(result_df[q].copy().to_numpy(), target, alpha=q)
#   pl_list.append(pl)
# print("Pinball: ", np.average(pl_list))
# print("Pinball: ", np.average(monthly_pl["pinball_mean"]))
print("MAPE: ", mape_val)
print("RMSE: ", rmse_val)
print("MAE: ", mae_val)


print()
print()
result_df["date"] = pd.to_datetime(result_df["timestamp"]).dt.date

daily_metrics = pd.DataFrame([
    {
        "date": date,
        "MAPE": mean_absolute_percentage_error(
            group["Price"], group["value"]
        ),
        "RMSE": root_mean_squared_error(
            group["Price"], group["value"]
        ),
        "MAE": mean_absolute_error(
            group["Price"], group["value"]
        ),
    }
    for date, group in result_df.groupby("date")
])

average_metrics = daily_metrics[["MAPE", "RMSE", "MAE"]].mean()

# print(daily_metrics)     # metric for each date
print(average_metrics)   # average across all dates

/var/folders/jp/nl85t4mj1118k5q8j1b1sdkm0000gn/T/ipykernel_17630/3400952150.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df["timestamp"] = pd.to_datetime(test_df["timestamp"])


MAPE:  0.3621726747102085
RMSE:  7.091191419511111
MAE:  5.018810462525962


MAPE    0.943586
RMSE    6.177371
MAE     5.018810
dtype: float64


In [48]:
result_df = result_df.rename(columns={"value":"LEAR_lmp"})
result_df.head()

,date,hour,LEAR_lmp,timestamp,Price
0,2025-01-01,0,47.389600,2025-01-01 00:00:00,42.43053
1,2025-01-01,1,46.262477,2025-01-01 01:00:00,42.49631
2,2025-01-01,2,45.317247,2025-01-01 02:00:00,42.19328
3,2025-01-01,3,45.363850,2025-01-01 03:00:00,41.71648
4,2025-01-01,4,45.530700,2025-01-01 04:00:00,42.43785


In [53]:

pred = pd.read_csv("lear_lmp_BANC.csv")
pred = pred.rename(columns={"Unnamed: 0": "date"})
pred["date"] = pd.to_datetime(pred["date"])

# Convert forecast_df's first row into the same wide format
first_row = forecast_df.iloc[[0]].copy()
first_row.index = pd.to_datetime(first_row.index)
first_row.index.name = "date"
first_row = first_row.reset_index()

# Add the missing first row
pred = (
    pd.concat([first_row, pred], ignore_index=True)
    .drop_duplicates(subset="date", keep="first")
    .sort_values("date")
)


df_long = pred.melt(
    id_vars="date",
    var_name="hour",
    value_name="Lear_lmp"
)

df_long["hour"] = (
    df_long["hour"]
    .str.extract(r"(\d+)", expand=False)
    .astype(int)
)

df_long["timestamp"] = (
    df_long["date"]
    + pd.to_timedelta(df_long["hour"], unit="h")
)

df_long = (
    df_long.sort_values("timestamp")
    .reset_index(drop=True)
)


# load ground truth
truth = pd.read_csv("datasets/lmp_NP15_data.csv")
truth = truth.rename(columns={"Unnamed: 0": "timestamp"})
truth["timestamp"] = pd.to_datetime(truth["timestamp"])

start = df_long["timestamp"].min()
end = pd.Timestamp("2026-01-01 00:00")

test_df = truth.loc[
    (truth["timestamp"] >= start)
    & (truth["timestamp"] < end)
].copy()

# merge with truth
result_df = df_long.merge(
    test_df[["timestamp", "Price"]],
    on="timestamp",
    how="inner"
)

result_df = result_df[
    ["date", "hour", "Lear_lmp", "timestamp", "Price"]
]

result_df

,date,hour,Lear_lmp,timestamp,Price
0,2024-12-31,0,35.141846,2024-12-31 00:00:00,42.44088
1,2024-12-31,1,36.644308,2024-12-31 01:00:00,46.01963
2,2024-12-31,2,32.172408,2024-12-31 02:00:00,42.83388
3,2024-12-31,3,36.222202,2024-12-31 03:00:00,42.51278
4,2024-12-31,4,36.137656,2024-12-31 04:00:00,45.39630
...,...,...,...,...,...
8779,2025-12-31,19,38.562076,2025-12-31 19:00:00,44.62684
8780,2025-12-31,20,40.021852,2025-12-31 20:00:00,40.63568
8781,2025-12-31,21,40.991889,2025-12-31 21:00:00,39.39058
8782,2025-12-31,22,40.749776,2025-12-31 22:00:00,38.03827


In [54]:
result_df.to_csv('lear_lmp_extra_day.csv')